# Phase 3: RAG Compliance Engine Testing Lab

This notebook tests the RAG-powered compliance engine we've built.

**What's Already Built:**
- ✅ `app/compliance.py` - ComplianceAuditor orchestrator
- ✅ `app/rag_query.py` - RAG query engine
- ✅ `app/rag_analyzer.py` - AI compliance analyzer
- ✅ `scripts/ingest_pdfs.py` - PDF ingestion pipeline
- ✅ `app/rules.py` - Hard rules validator

**This Notebook:**
- Tests the complete audit workflow
- Experiments with different invoice scenarios
- Debugs issues in isolation

In [ ]:
# Cell 1: Setup and Imports
import sys
import os

# Get the micro-cfo directory (parent of .agents)
notebook_dir = os.getcwd()
if notebook_dir.endswith('.agents'):
    micro_cfo_dir = os.path.dirname(notebook_dir)
else:
    micro_cfo_dir = notebook_dir

# Add micro-cfo directory to path
sys.path.insert(0, micro_cfo_dir)

from dotenv import load_dotenv
from app.schemas import InvoiceData, ExpenseCategory
from app.compliance import ComplianceAuditor
from app.rules import validate_gstin, validate_gst_rate

# Load environment variables from micro-cfo/.env
env_path = os.path.join(micro_cfo_dir, '.env')
load_dotenv(env_path)

api_key = os.getenv("GOOGLE_API_KEY")
convex_url = os.getenv("CONVEX_URL")

print(f"📁 Working directory: {micro_cfo_dir}")
print(f"📄 Loading .env from: {env_path}")
print(f"📄 .env exists: {os.path.exists(env_path)}\n")

if not api_key:
    print("❌ Error: GOOGLE_API_KEY not found in .env")
else:
    print("✅ API Key Loaded!")

if not convex_url:
    print("❌ Error: CONVEX_URL not found in .env")
else:
    print("✅ Convex URL Loaded!")

print("\n📦 All modules imported successfully!")

In [ ]:
# Cell 2: Test Hard Rules Validator
print("🧪 Testing Hard Rules Validator\n")

# Test GSTIN validation
valid_gstin = "27AAPFU0939F1ZV"
invalid_gstin = "INVALID123"

print(f"Valid GSTIN '{valid_gstin}': {validate_gstin(valid_gstin)}")
print(f"Invalid GSTIN '{invalid_gstin}': {validate_gstin(invalid_gstin)}")

# Test GST rate validation
print("\n📊 GST Rate Validation:")
print(f"18% rate (118 total, 18 tax): {validate_gst_rate(118.0, 18.0)}")
print(f"10% rate (110 total, 10 tax): {validate_gst_rate(110.0, 10.0)}")
print(f"Zero total: {validate_gst_rate(0.0, 0.0)}")

In [ ]:
# Cell 3: Create Test Invoices
print("📝 Creating Test Invoices\n")

# Test Case 1: Compliant Office Supplies
invoice_compliant = InvoiceData(
    vendor_name="Office Mart",
    invoice_number="INV-001",
    date="2025-02-10",
    total_amount=1180.0,
    tax_amount=180.0,  # 18% GST
    gstin="27AAPFU0939F1ZV",
    category=ExpenseCategory.OFFICE_SUPPLIES,
    item_description="Printer paper and stationery"
)

# Test Case 2: Blocked Food & Beverage
invoice_blocked = InvoiceData(
    vendor_name="Hotel Taj",
    invoice_number="INV-002",
    date="2025-02-10",
    total_amount=2360.0,
    tax_amount=360.0,  # 18% GST
    gstin="27BBBBB1234B1Z5",
    category=ExpenseCategory.FOOD_BEVERAGE,
    item_description="Staff lunch at hotel restaurant"
)

# Test Case 3: High-Value Cash Transaction
invoice_cash_limit = InvoiceData(
    vendor_name="Electronics Store",
    invoice_number="INV-003",
    date="2025-02-10",
    total_amount=15000.0,
    tax_amount=2700.0,  # 18% GST
    gstin="27CCCCC5678C1Z5",
    category=ExpenseCategory.ELECTRONICS,
    item_description="Laptop for office use"
)

# Test Case 4: Invalid GSTIN
invoice_invalid_gstin = InvoiceData(
    vendor_name="Local Vendor",
    total_amount=590.0,
    tax_amount=90.0,
    gstin="INVALID",
    category=ExpenseCategory.OFFICE_SUPPLIES,
    item_description="Office supplies"
)

print("✅ Test invoices created!")
print(f"  1. Compliant: {invoice_compliant.vendor_name}")
print(f"  2. Blocked ITC: {invoice_blocked.vendor_name}")
print(f"  3. Cash Limit: {invoice_cash_limit.vendor_name}")
print(f"  4. Invalid GSTIN: {invoice_invalid_gstin.vendor_name}")

In [ ]:
# Cell 4: Initialize Compliance Auditor
print("🚀 Initializing Compliance Auditor...\n")

try:
    auditor = ComplianceAuditor()
    print("✅ ComplianceAuditor initialized successfully!")
    print(f"   - Hard Rules Validator: Ready")
    print(f"   - RAG Query Engine: Ready")
    print(f"   - AI Compliance Analyzer: Ready")
except Exception as e:
    print(f"❌ Error initializing auditor: {e}")

In [ ]:
# Cell 5: Test Case 1 - Compliant Invoice
print("\n" + "="*60)
print("🧪 TEST CASE 1: Compliant Office Supplies")
print("="*60)

try:
    result = auditor.audit_invoice(invoice_compliant)
    
    print(f"\n📊 Audit Result:")
    print(f"   Status: {result['status']}")
    print(f"   Category: {result['category']}")
    print(f"   ITC Eligible: {result.get('itc_eligible', 'N/A')}")
    
    if result['flags']:
        print(f"\n⚠️ Flags:")
        for flag in result['flags']:
            print(f"   - {flag}")
    else:
        print(f"\n✅ No compliance issues found!")
    
    if result.get('citations'):
        print(f"\n📚 Legal Citations:")
        for citation in result['citations']:
            print(f"   - {citation['source']}, Page {citation['page']}")
            
except Exception as e:
    print(f"\n❌ Error during audit: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 6: Test Case 2 - Blocked Food & Beverage
print("\n" + "="*60)
print("🧪 TEST CASE 2: Blocked Food & Beverage")
print("="*60)

try:
    result = auditor.audit_invoice(invoice_blocked)
    
    print(f"\n📊 Audit Result:")
    print(f"   Status: {result['status']}")
    print(f"   Category: {result['category']}")
    print(f"   ITC Eligible: {result.get('itc_eligible', 'N/A')}")
    
    if result['flags']:
        print(f"\n🚫 Flags:")
        for flag in result['flags']:
            print(f"   - {flag}")
    
    if result.get('citations'):
        print(f"\n📚 Legal Citations:")
        for citation in result['citations']:
            print(f"   - {citation['source']}, Page {citation['page']}")
            
except Exception as e:
    print(f"\n❌ Error during audit: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 7: Test Case 3 - High-Value Transaction
print("\n" + "="*60)
print("🧪 TEST CASE 3: High-Value Cash Transaction")
print("="*60)

try:
    result = auditor.audit_invoice(invoice_cash_limit)
    
    print(f"\n📊 Audit Result:")
    print(f"   Status: {result['status']}")
    print(f"   Category: {result['category']}")
    print(f"   ITC Eligible: {result.get('itc_eligible', 'N/A')}")
    
    if result['flags']:
        print(f"\n⚠️ Flags:")
        for flag in result['flags']:
            print(f"   - {flag}")
    
    if result.get('citations'):
        print(f"\n📚 Legal Citations:")
        for citation in result['citations']:
            print(f"   - {citation['source']}, Page {citation['page']}")
            
except Exception as e:
    print(f"\n❌ Error during audit: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 8: Test Case 4 - Invalid GSTIN
print("\n" + "="*60)
print("🧪 TEST CASE 4: Invalid GSTIN")
print("="*60)

try:
    result = auditor.audit_invoice(invoice_invalid_gstin)
    
    print(f"\n📊 Audit Result:")
    print(f"   Status: {result['status']}")
    print(f"   Category: {result['category']}")
    print(f"   ITC Eligible: {result.get('itc_eligible', 'N/A')}")
    
    if result['flags']:
        print(f"\n❌ Flags:")
        for flag in result['flags']:
            print(f"   - {flag}")
    
    if result.get('citations'):
        print(f"\n📚 Legal Citations:")
        for citation in result['citations']:
            print(f"   - {citation['source']}, Page {citation['page']}")
            
except Exception as e:
    print(f"\n❌ Error during audit: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 9: Test RAG Query Generation
print("\n" + "="*60)
print("🔍 Testing RAG Query Generation")
print("="*60)

from app.rag_query import RAGQueryEngine

query_engine = RAGQueryEngine(api_key)

# Test query generation for different invoice types
test_invoices = [
    invoice_compliant,
    invoice_blocked,
    invoice_cash_limit
]

for i, invoice in enumerate(test_invoices, 1):
    query = query_engine.generate_search_query(invoice)
    print(f"\n{i}. {invoice.vendor_name}:")
    print(f"   Query: {query}")

In [ ]:
# Cell 10: Custom Invoice Testing
print("\n" + "="*60)
print("🎯 Custom Invoice Testing")
print("="*60)
print("\nCreate your own invoice and test it!\n")

# Modify these values to test different scenarios
custom_invoice = InvoiceData(
    vendor_name="Your Vendor Name",
    invoice_number="CUSTOM-001",
    date="2025-02-10",
    total_amount=1000.0,
    tax_amount=180.0,
    gstin="27AAPFU0939F1ZV",  # Change to test GSTIN validation
    category=ExpenseCategory.OFFICE_SUPPLIES,  # Try: FOOD_BEVERAGE, TRAVEL, etc.
    item_description="Your item description here"
)

print(f"Testing: {custom_invoice.vendor_name}")
print(f"Category: {custom_invoice.category.value}")
print(f"Amount: ₹{custom_invoice.total_amount}\n")

try:
    result = auditor.audit_invoice(custom_invoice)
    
    print(f"📊 Result: {result['status'].upper()}")
    
    if result['flags']:
        print(f"\nFlags:")
        for flag in result['flags']:
            print(f"  {flag}")
    
    if result.get('citations'):
        print(f"\nCitations:")
        for citation in result['citations']:
            print(f"  📚 {citation['source']}, Page {citation['page']}")
            
except Exception as e:
    print(f"❌ Error: {e}")

## Summary

This notebook tests the complete RAG Compliance Engine:

1. **Hard Rules Validation** - GSTIN format, GST rates, cash limits
2. **RAG Query Generation** - Context-aware search queries
3. **Vector Search** - Retrieves relevant legal text from Convex
4. **AI Analysis** - Gemini analyzes compliance with legal context
5. **Report Generation** - Combines all results with citations

**Next Steps:**
- Run the PDF ingestion script: `python scripts/ingest_pdfs.py`
- Integrate with Telegram bot in `bot.py`
- Continue with spec tasks 7-17